<a href="https://colab.research.google.com/github/mp371366/ML/blob/main/LAB13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dekonwolucja

## Czym jest konwolucja transponowana?
Konwolucja transponowana (ang. transposed convolution, znana również jako fractionally-strided convolution lub potocznie, choć matematycznie nieprecyzyjnie: dekonwolucja) to operacja warstwy sieci neuronowej, która służy do zwiększania wymiarów przestrzennych (upsamplingu) map cech.

W architekturach typu Autoenkoder lub U-Net, enkoder kompresuje obraz wejściowy, zmniejszając jego rozdzielczość i wyciągając kluczowe cechy. Konwolucja transponowana działa w dekoderze jako odwrotność tego procesu pod względem kształtu – przyjmuje małą, skompresowaną mapę cech i przekształca ją w macierz o większej rozdzielczości, próbując zrekonstruować szczegóły przestrzenne (np. oryginalny obraz lub maskę segmentacji).

## Czym różni się od zwykłej konwolucji?

Główna różnica tkwi w kierunku przekształcenia matematycznego i relacji między pikselami wejściowymi a wyjściowymi:

Zwykła konwolucja (Relacja "Wielu do Jednego" / Many-to-One): Warstwa nakłada filtr (kernel) na określony obszar wejściowy, wykonuje operację iloczynu skalarnego elementów filtra i obrazu, a następnie zwraca jedną, pojedynczą wartość na mapie wyjściowej. Efektem tego jest zazwyczaj zmniejszenie wymiarów przestrzennych obrazu.

Konwolucja transponowana (Relacja "Jeden do Wielu" / One-to-Many): Warstwa bierze jedną, pojedynczą wartość z mapy wejściowej, mnoży ją przez wszystkie wagi znajdujące się w filtrze (kernelu) i ten przeskalowany filtr (całą macierz wartości) "rozprasza" i rzutuje na mapę wyjściową. W ten sposób jeden piksel generuje cały region na wyjściu, co prowadzi do zwiększenia wymiarów przestrzennych.

## W jaki sposób zwiększa ona rozdzielczość map cech?

Proces upsamplingu (podnoszenia rozdzielczości) krok po kroku wygląda następująco:
- Algorytm pobiera pierwszy piksel (wartość skalarorwą) z lewego górnego rogu macierzy wejściowej.
- Mnoży tę wartość przez każdą wagę wewnątrz filtra o rozmiarze $k \times k$. Otrzymuje w ten sposób nową macierz pośrednią o rozmiarze filtra.
- Zapisuje tę macierz pośrednią w lewym górnym rogu nowej, większej siatki wyjściowej.Następnie przechodzi do kolejnego piksela wejściowego i wykonuje to samo mnożenie przez filtr.
- Wynik tego mnożenia jest rzutowany na siatkę wyjściową, ale przesunięty o wartość zdefiniowaną przez parametr kroku (stride).
- Kluczowy krok: Ponieważ obszary rzutowania poszczególnych pikseli mogą się na siebie nakładać, wartości w miejscach nałożenia się filtrów na siatce wyjściowej są do siebie dodawane (sumowane). Proces ten powtarza się dla wszystkich pikseli wejściowych, tworząc gęstą, dużą mapę wyjściową.

##  Czym są stride, padding oraz kernel size i jak wpływają na wynik konwolucji transponowanej?

W konwolucji transponowanej parametry te działają inaczej (często intuicyjnie odwrotnie) niż w standardowej konwolucji:

- Rozmiar jądra / filtra (Kernel size, $k$): Definiuje rozmiar obszaru na wyjściu, na który "promieniuje" pojedynczy piksel wejściowy. Im większy jest kernel, tym większy obszar wyjściowy pokrywa jeden punkt z wejścia i tym większa staje się cała mapa wyjściowa.
- Krok (Stride, $s$): W zwykłej konwolucji stride oznacza przesunięcie filtra na wejściu. W konwolucji transponowanej stride definiuje, o ile pikseli przesuwamy rzutowany filtr na siatce WYJŚCIOWEJ, kiedy przesuniemy się o 1 piksel na wejściu. Ustawienie $s > 1$ powoduje "rozstrzelenie" rzutów dalej od siebie, co jest głównym mechanizmem zwiększania rozdzielczości (np. $s=2$ w przybliżeniu podwaja wymiary obrazu). Jeśli $s > 1$, między rzutowanymi jądrami na wyjściu mogą pojawić się puste przestrzenie (zera), które są uzupełniane w trakcie operacji.
- Margines (Padding, $p$): W zwykłej konwolucji padding sztucznie powiększa wejście. W konwolucji transponowanej padding oznacza obcięcie (wykadrowanie) zewnętrznych krawędzi z finalnie wygenerowanej mapy wyjściowej. Dlatego zwiększenie wartości padding w kodzie (np. w PyTorch `nn.ConvTranspose2d`) sprawi, że obraz wyjściowy będzie mniejszy, a nie większy.

## Rozmiaru wyjściowy

$$O = (I - 1) \times s + k - 2p$$

- $O$ (Output size): Rozmiar wyjściowy
- $I$ (Input size): Rozmiar wejściowy
- $s$ (Stride): Krok
- $k$ (Kernel size): Rozmiar filtra (jądra)
- $p$ (Padding): Margines


## Obrazek

<img src="https://raw.githubusercontent.com/mp371366/ML/main/pic.jpg" style="transform: rotate(-90deg);" alt="Dekonwolucja"/>